# Final Assignment: Analyzing Historical Stock/Revenue Data and Building a Dashboard

Extracting Tesla and GameStop stock data (yfinance) and revenue data (web scraping), then building dashboards.

In [ ]:
!pip install yfinance==0.2.38 pandas==2.2.2 nbformat beautifulsoup4 lxml html5lib plotly requests

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
def make_graph(stock_data, revenue_data, stock):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=("Historical Share Price", "Historical Revenue"),
                        vertical_spacing=.3)
    stock_data_specific = stock_data[stock_data.Date <= '2021-06-14']
    revenue_data_specific = revenue_data[revenue_data.Date <= '2021-04-30']
    fig.add_trace(go.Scatter(x=pd.to_datetime(stock_data_specific.Date),
                             y=stock_data_specific.Close.astype("float"),
                             name="Share Price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=pd.to_datetime(revenue_data_specific.Date),
                             y=revenue_data_specific.Revenue.astype("float"),
                             name="Revenue"), row=2, col=1)
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)
    fig.update_layout(showlegend=False, height=900, title=stock, xaxis_rangeslider_visible=True)
    fig.show()

### Helper to scrape a quarterly revenue table with BeautifulSoup

In [ ]:
def get_revenue(url, title):
    html_data = requests.get(url).text
    soup = BeautifulSoup(html_data, 'html.parser')
    candidates = [t for t in soup.find_all('table') if title in t.get_text()]
    table = min(candidates, key=lambda t: len(t.get_text()))  # innermost matching table
    rows = []
    for tr in table.find_all('tr'):
        tds = tr.find_all('td')
        if len(tds) == 2:
            rows.append({'Date': tds[0].get_text(strip=True), 'Revenue': tds[1].get_text(strip=True)})
    df = pd.DataFrame(rows)
    df['Revenue'] = df['Revenue'].str.replace(r'[$,]', '', regex=True)
    df = df[df['Revenue'] != ''].reset_index(drop=True)
    return df

## Question 1 - Extracting Tesla Stock Data Using yfinance

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.head()

## Question 2 - Extracting Tesla Revenue Data Using Webscraping

In [ ]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DA0321EN-SkillsNetwork/labs/project/revenue.htm"
tesla_revenue = get_revenue(url, "Tesla Quarterly Revenue")
tesla_revenue.tail()

## Question 3 - Extracting GameStop Stock Data Using yfinance

In [ ]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.head()

## Question 4 - Extracting GameStop Revenue Data Using Webscraping

In [ ]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DA0321EN-SkillsNetwork/labs/project/stock.html"
gme_revenue = get_revenue(url, "GameStop Quarterly Revenue")
gme_revenue.tail()

## Question 5 - Tesla Stock and Revenue Dashboard

In [ ]:
make_graph(tesla_data, tesla_revenue, 'Tesla')

## Question 6 - GameStop Stock and Revenue Dashboard

In [ ]:
make_graph(gme_data, gme_revenue, 'GameStop')